# Extract Range of Motion Data from Archived Results

This notebook extracts minimum and maximum muscle lengths from all simulation folders in `archived_results/` and compiles them into a single CSV file.

In [ ]:
import os
import re
import pandas as pd
from pathlib import Path

In [ ]:
# Define the base directory
base_dir = Path('archived_results')
output_file = 'rom_data_all_combinations.csv'

# Check if archived_results directory exists
if not base_dir.exists():
    raise FileNotFoundError(f"Directory {base_dir} not found")

print(f"Base directory: {base_dir.absolute()}")
print(f"Output file: {output_file}")

In [ ]:
def parse_folder_name(folder_name):
    """
    Parse folder name to extract parameters.
    Format: F{force}_y{muscle_y}_Vol{volume}_Am{am}_rho{rho}
    Example: F00_y32.69_Vol421.6_Am450_rho10.493
    """
    pattern = r'F(\d+)_y([\d.]+)_Vol([\d.]+)_Am(\d+)_rho([\d.]+)'
    match = re.match(pattern, folder_name)
    
    if match:
        return {
            'force_N': float(match.group(1)),
            'muscle_extent_y_cm': float(match.group(2)),
            'volume_cm3': float(match.group(3)),
            'am_cm_inv': float(match.group(4)),
            'rho_1e4_kg_cm3': float(match.group(5))
        }
    else:
        return None

def extract_rom_data(rom_file_path):
    """
    Extract min and max muscle length from range_of_motion.txt file.
    """
    try:
        with open(rom_file_path, 'r') as f:
            content = f.read()
        
        # Extract z_max
        z_max_match = re.search(r'Maximum muscle length \(z_max\):\s+([\d.]+)\s+cm', content)
        # Extract z_min
        z_min_match = re.search(r'Minimum muscle length \(z_min\):\s+([\d.]+)\s+cm', content)
        # Extract ROM
        rom_match = re.search(r'Range of Motion \(ROM\):\s+([\d.]+)\s+cm', content)
        
        if z_max_match and z_min_match and rom_match:
            return {
                'z_max_cm': float(z_max_match.group(1)),
                'z_min_cm': float(z_min_match.group(1)),
                'rom_cm': float(rom_match.group(1))
            }
        else:
            return None
    except Exception as e:
        print(f"Error reading {rom_file_path}: {e}")
        return None

# Test parsing on a sample folder name
test_folder = "F00_y32.69_Vol421.6_Am450_rho10.493"
print(f"Test parsing: {test_folder}")
print(parse_folder_name(test_folder))

In [ ]:
# Iterate through all folders in archived_results
data_list = []
folders_processed = 0
folders_failed = 0

for folder in sorted(base_dir.iterdir()):
    if folder.is_dir():
        # Parse folder name to get parameters
        params = parse_folder_name(folder.name)
        
        if params is None:
            print(f"Warning: Could not parse folder name: {folder.name}")
            folders_failed += 1
            continue
        
        # Look for range_of_motion.txt file
        rom_file = folder / 'range_of_motion.txt'
        
        if rom_file.exists():
            rom_data = extract_rom_data(rom_file)
            
            if rom_data is not None:
                # Combine parameters and ROM data
                row_data = {**params, **rom_data}
                row_data['folder_name'] = folder.name
                data_list.append(row_data)
                folders_processed += 1
            else:
                print(f"Warning: Could not extract ROM data from: {rom_file}")
                folders_failed += 1
        else:
            print(f"Warning: range_of_motion.txt not found in: {folder.name}")
            folders_failed += 1

print(f"\nFolders processed successfully: {folders_processed}")
print(f"Folders failed: {folders_failed}")

In [ ]:
# Create DataFrame
df = pd.DataFrame(data_list)

# Reorder columns for better readability
column_order = [
    'folder_name',
    'force_N',
    'muscle_extent_y_cm',
    'volume_cm3',
    'am_cm_inv',
    'rho_1e4_kg_cm3',
    'z_max_cm',
    'z_min_cm',
    'rom_cm'
]

df = df[column_order]

# Display summary
print(f"\nTotal rows: {len(df)}")
print(f"\nFirst few rows:")
df.head(10)

In [ ]:
# Display summary statistics
print("\nParameter ranges:")
print(f"Force: {df['force_N'].min()} - {df['force_N'].max()} N")
print(f"Muscle extent y: {df['muscle_extent_y_cm'].min()} - {df['muscle_extent_y_cm'].max()} cm")
print(f"Volume: {df['volume_cm3'].min()} - {df['volume_cm3'].max()} cm³")
print(f"Am: {df['am_cm_inv'].min()} - {df['am_cm_inv'].max()} cm⁻¹")
print(f"Rho: {df['rho_1e4_kg_cm3'].min()} - {df['rho_1e4_kg_cm3'].max()} 1e-4 kg/cm³")
print(f"\nROM ranges:")
print(f"z_max: {df['z_max_cm'].min()} - {df['z_max_cm'].max()} cm")
print(f"z_min: {df['z_min_cm'].min()} - {df['z_min_cm'].max()} cm")
print(f"ROM: {df['rom_cm'].min()} - {df['rom_cm'].max()} cm")

df.describe()

In [ ]:
# Save to CSV
df.to_csv(output_file, index=False)
print(f"\nData saved to: {output_file}")
print(f"Absolute path: {Path(output_file).absolute()}")

In [ ]:
# Verify unique combinations
print("\nUnique values per parameter:")
print(f"Forces: {sorted(df['force_N'].unique())}")
print(f"Muscle Y: {sorted(df['muscle_extent_y_cm'].unique())}")
print(f"Volumes: {sorted(df['volume_cm3'].unique())}")
print(f"Am values: {sorted(df['am_cm_inv'].unique())}")
print(f"Rho values: {sorted(df['rho_1e4_kg_cm3'].unique())}")